# SVM Model with scikit-learn


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

from student_depression.models import LinearSVMDual

In [ ]:
df_val = pd.read_csv("../data/processed/test.csv")
df_train = pd.read_csv("../data/processed/train.csv")
X_val = df_val.drop("Depression", axis=1)
y_val = df_val["Depression"]
X = df_train.drop("Depression", axis=1)
y = df_train["Depression"]

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

# Create pipeline with MinMaxScaler and SVM
pipe = Pipeline([("scaler", MinMaxScaler()), ("svm", SVC(kernel="linear", random_state=42))])

# Define parameter grid for pipeline
param_grid = {"svm__C": [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]}

# Use pipeline in GridSearchCV
grid_search = GridSearchCV(pipe, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X, y)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation accuracy:", grid_search.best_score_)

# Use best model and evaluate on validation set
best_model = grid_search.best_estimator_
val_predictions = best_model.predict(X_val)

# Print validation results
print("\nValidation Set Results:")
print("Accuracy on validation set:", accuracy_score(y_val, val_predictions))
print("\nClassification Report on Validation Set:")
print(classification_report(y_val, val_predictions, digits=4))

In [ ]:
import os

import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


# After getting predictions and running classification_report
def save_metrics_to_csv(model_name, y_true, y_pred, filepath="../results/model_metrics.csv"):
    """
    Save model metrics to CSV file with each model as a row

    Parameters:
    -----------
    model_name : str
        Name of the model
    y_true : array-like
        True labels
    y_pred : array-like
        Predicted labels
    filepath : str
        Path to CSV file
    """
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)

    # Class 0 metrics
    precision_0 = precision_score(y_true, y_pred, pos_label=0)
    recall_0 = recall_score(y_true, y_pred, pos_label=0)
    f1_0 = f1_score(y_true, y_pred, pos_label=0)

    # Class 1 metrics
    precision_1 = precision_score(y_true, y_pred, pos_label=1)
    recall_1 = recall_score(y_true, y_pred, pos_label=1)
    f1_1 = f1_score(y_true, y_pred, pos_label=1)

    # Average metrics
    precision_avg = precision_score(y_true, y_pred, average="macro")
    recall_avg = recall_score(y_true, y_pred, average="macro")
    f1_avg = f1_score(y_true, y_pred, average="macro")

    # Create a dictionary with all metrics
    metrics_dict = {
        "model": model_name,
        "accuracy": accuracy,
        "precision_class0": precision_0,
        "recall_class0": recall_0,
        "f1_class0": f1_0,
        "precision_class1": precision_1,
        "recall_class1": recall_1,
        "f1_class1": f1_1,
        "precision_avg": precision_avg,
        "recall_avg": recall_avg,
        "f1_avg": f1_avg,
    }

    # Check if file exists
    if os.path.exists(filepath):
        # Read existing data and append new row
        metrics_df = pd.read_csv(filepath)

        # Check if model already exists in the dataframe
        if model_name in metrics_df["model"].values:
            # Update existing row
            metrics_df.loc[metrics_df["model"] == model_name] = pd.Series(metrics_dict)
        else:
            # Append new row
            metrics_df = pd.concat([metrics_df, pd.DataFrame([metrics_dict])], ignore_index=True)
    else:
        # Create new dataframe
        metrics_df = pd.DataFrame([metrics_dict])

    # Save to CSV
    metrics_df.to_csv(filepath, index=False)
    print(f"Metrics saved to {filepath}")

    return metrics_df


# Use the function after evaluating your model
# Example usage after running the model:
save_metrics_to_csv("SVM_Linear", y_val, val_predictions)

## Custom SVM Implementation (from scratch)

### Train and Evaluate Custom SVM

In [ ]:
# Định nghĩa siêu tham số
C_value = 1.0  # C nên chọn từ kết quả GridSearchCV tốt nhất
tol_value = 1e-3  # Tolerance thường dùng 1e-4 hoặc 1e-3
max_iter_value = 1000  # Số vòng lặp tối đa
X = X.head(3000)  # Chọn 3000 mẫu đầu tiên
y = y.head(3000)  # Chọn 3000 nhãn đầu tiên
X_train_array = X.to_numpy()
y_train_array = y.to_numpy()
# Huấn luyện và đánh giá mô hình
svm_scratch = LinearSVMDual(C=C_value, tolerance=tol_value, max_iter=max_iter_value)
svm_scratch.fit(X_train_array, y_train_array)  # Sử dụng toàn bộ training data

# Đánh giá trên tập validation
y_pred_scratch = svm_scratch.predict(X_val.to_numpy())
print("Accuracy (scratch):", accuracy_score(y_val, y_pred_scratch))
print("\nClassification Report:")
print(classification_report(y_val, y_pred_scratch, digits=4))

In [ ]:
save_metrics_to_csv("SVM_Linear_scratch", y_val, y_pred_scratch)